In [1]:
# ==========================================
# Lesson 29 - Sampling Satellite Data
# ==========================================

import ee
import geemap

ee.Initialize(project="oc-flux")

Map = geemap.Map()

point = ee.Geometry.Point([77.2090,28.6139])

collection = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(point)
    .filterDate("2023-01-01","2023-12-31")
)

image = collection.median()

Map.centerObject(point,10)

Map

Map(center=[28.613900000000005, 77.209], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

In [ ]:
Creating a Sample Point
You should see a red point on the map.

In [2]:
sample_point = ee.Geometry.Point(
    [77.2090,28.6139]
)

Map.addLayer(
    sample_point,
    {"color":"red"},
    "Sample Point"
)

Map

Map(bottom=27634.0, center=[28.586330559329642, 77.74200439453125], controls=(WidgetControl(options=['position…

In [ ]:
Extract Pixel Values

Use sample().
This extracts the Landsat band values at the selected location.

In [3]:
sample = image.sample(

    region=sample_point,

    scale=30,

    geometries=True

)

print(sample.getInfo())

{'type': 'FeatureCollection', 'columns': {'QA_PIXEL': 'Float<0.0, 65535.0>', 'QA_RADSAT': 'Float<0.0, 65535.0>', 'SR_B1': 'Float<0.0, 65535.0>', 'SR_B2': 'Float<0.0, 65535.0>', 'SR_B3': 'Float<0.0, 65535.0>', 'SR_B4': 'Float<0.0, 65535.0>', 'SR_B5': 'Float<0.0, 65535.0>', 'SR_B6': 'Float<0.0, 65535.0>', 'SR_B7': 'Float<0.0, 65535.0>', 'SR_QA_AEROSOL': 'Float<0.0, 255.0>', 'ST_ATRAN': 'Float<-32768.0, 32767.0>', 'ST_B10': 'Float<0.0, 65535.0>', 'ST_CDIST': 'Float<-32768.0, 32767.0>', 'ST_DRAD': 'Float<-32768.0, 32767.0>', 'ST_EMIS': 'Float<-32768.0, 32767.0>', 'ST_EMSD': 'Float<-32768.0, 32767.0>', 'ST_QA': 'Float<-32768.0, 32767.0>', 'ST_TRAD': 'Float<-32768.0, 32767.0>', 'ST_URAD': 'Float<-32768.0, 32767.0>'}, 'properties': {'band_order': ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT']}, 'features': [{'type': 'Feature', 'geometry':

In [ ]:
Understanding the Output

The result is a FeatureCollection.

Example:

FeatureCollection

↓

Feature

↓

Properties

↓

SR_B2 = 10342

SR_B3 = 11185

SR_B4 = 9876

SR_B5 = 7650

These are the reflectance values for that pixel.

In [ ]:
Sample Multiple Points

In [4]:
points = ee.FeatureCollection([

ee.Feature(
    ee.Geometry.Point([77.20,28.61])
),

ee.Feature(
    ee.Geometry.Point([77.22,28.63])
),

ee.Feature(
    ee.Geometry.Point([77.24,28.60])
)

])

In [ ]:
Sample Regions

In [5]:
training = image.sampleRegions(

    collection=points,

    scale=30,

    geometries=True

)

print(
    training.size().getInfo()
)

3


In [ ]:
Visualize the Points
What Does Each Record Contain?

Each sampled point contains:

Geometry

↓

SR_B2

↓

SR_B3

↓

SR_B4

↓

SR_B5

↓

Coordinates

Later, you will also add:

POC

In [6]:
Map.addLayer(
    points,
    {"color":"yellow"},
    "Training Points"
)

Map

Map(bottom=437555.0, center=[28.611349652389134, 77.22581863403322], controls=(WidgetControl(options=['positio…

In [ ]:
Adding POC Measurements

Imagine field measurements:

Point	POC (mg/L)
1	2.8
2	4.6
3	6.2

Attach these values:
Each point now has both location and measured POC.

In [7]:
points = ee.FeatureCollection([

ee.Feature(
    ee.Geometry.Point([77.20,28.61]),
    {"POC":2.8}
),

ee.Feature(
    ee.Geometry.Point([77.22,28.63]),
    {"POC":4.6}
),

ee.Feature(
    ee.Geometry.Point([77.24,28.60]),
    {"POC":6.2}
)

])

In [ ]:
SampleRegions with Field Data
This creates a training dataset containing:

Satellite bands
POC measurements

In [8]:
training = image.sampleRegions(

    collection=points,

    properties=["POC"],

    scale=30

)

In [ ]:
Inspect the Training Data

In [9]:
print(
    training.first().getInfo()
)

{'type': 'Feature', 'geometry': None, 'id': '0_0', 'properties': {'POC': 2.8, 'QA_PIXEL': 21824, 'QA_RADSAT': 0, 'SR_B1': 9479, 'SR_B2': 10050.5, 'SR_B3': 11066, 'SR_B4': 11118.5, 'SR_B5': 16316.5, 'SR_B6': 13612.5, 'SR_B7': 11934.5, 'SR_QA_AEROSOL': 224, 'ST_ATRAN': 7716.5, 'ST_B10': 43700.5, 'ST_CDIST': 40.5, 'ST_DRAD': 926.5, 'ST_EMIS': 9608, 'ST_EMSD': 169, 'ST_QA': 435.5, 'ST_TRAD': 8687.5, 'ST_URAD': 1893.5}}


In [ ]:
| Function          | Purpose                                           |
| ----------------- | ------------------------------------------------- |
| `sample()`        | Extract pixels from a region or point             |
| `sampleRegions()` | Extract pixels for many points/features           |
| `reduceRegion()`  | Summarize values for one region (mean, max, etc.) |
| `reduceRegions()` | Summarize values for many regions                 |


In [ ]:
Using reduceRegion()

Calculate the average Red band value within a buffer.
This returns the mean reflectance over the buffered area.

In [10]:
result = image.select("SR_B4").reduceRegion(

    reducer=ee.Reducer.mean(),

    geometry=sample_point.buffer(500),

    scale=30

)

print(result.getInfo())

{'SR_B4': 12161.969176799625}


In [ ]:
Using reduceRegions()

Suppose you have several polygons.
Polygon 1

Polygon 2

Polygon 3

Earth Engine computes statistics for each polygon.

Example:
This is useful for averaging values over lakes, river reaches, or administrative boundaries.

In [11]:
# Example only

# results = image.reduceRegions(
#     collection=polygons,
#     reducer=ee.Reducer.mean(),
#     scale=30
# )